In [19]:
pretext = "Çiğ köfteyi çok seven Şule, öğle yemeğinde ılık süt içti."
unicode = [ord(x) for x in pretext]
text = list( pretext.encode("UTF-8"))
print(len(unicode))
print(len(text))

pretext = "Şükrü, yağışlı bir öğleden sonra küçük köyün çiçekli bahçesinde ıhlamur içti."
unicode = [ord(x) for x in pretext]
text = list( pretext.encode("UTF-8"))
print(len(unicode))
print(len(text))


pretext = "Çağlar"
unicode = [ord(x) for x in pretext]
text = list( pretext.encode("UTF-8"))
print(unicode)
print(text)

57
69
77
96
[199, 97, 287, 108, 97, 114]
[195, 135, 97, 196, 159, 108, 97, 114]


In [ ]:
with open("input.txt", 'r') as f:
    text = f.read()

en_text = list(text.encode("utf-8"))
vocab = {i: bytes([i]) for i in range(256)}

def find_maxpair(en_text):
    pairs = {}
    for i, j in zip(en_text, en_text[1:]):
        if (i, j) not in pairs:
            pairs[(i, j)] = 1
        else:
            pairs[(i, j)] += 1
    return pairs, max(pairs, key=pairs.get) 

def squeze(text):
    z = len(text)
    if z < 2:
        return text
    i = 0
    new_text = []
    pairs, max_pair = find_maxpair(text)
    #print(sorted(((v, k) for k, v in pairs.items()), reverse=True)[:20])

    if(pairs[max_pair] == 1):
        return text

    new_id = len(vocab)
    id, id_next = max_pair
    vocab[new_id] = vocab[id] + vocab[id_next]
    while(i < z):
        
        if(i<z-1 and max_pair == (text[i], text[i +1])):
            new_text.append(new_id)
            i += 2
        else:
            new_text.append(text[i])
            i += 1
    return new_text

vocab_size = 276
num_merge = vocab_size - 256
merges ={}
new_text = en_text

for i in range(num_merge):
    pairs, max_pair = find_maxpair(new_text)
    new_id = len(vocab)
    id, id_next = max_pair
    vocab[new_id] = max_pair[0] + max_pair[1]
    merges[max_pair] = vocab_size + i
    #print(f"Tokens {max_pair} merged into {pairs[max_pair]}")
    new_text = squeze(new_text)
print(len(en_text)/len(new_text))
print(len(en_text))
print(len(new_text))

[((196, 177), 256), ((110, 32), 257), ((195, 188), 258), ((97, 114), 259), ((101, 32), 260), ((197, 159), 261), ((46, 32), 262), ((101, 114), 263), ((97, 32), 264), ((44, 32), 265)]


In [28]:
def decode(ids):
    tokens = b"".join(vocab[t] for t in ids)
    return tokens.decode("utf-8", errors="replace")

harfler = {idx: b.decode("utf-8", errors="replace") for idx, b in vocab.items()}

def merge(tokens, pair, new_id):
    new_tokens = []
    i = 0
    while i < len(tokens):
        if i < len(tokens) - 1 and (tokens[i], tokens[i + 1]) == pair:
            new_tokens.append(new_id)
            i += 2
        else:
            new_tokens.append(tokens[i])
            i += 1
    return new_tokens

def encode(text):
    tokens = list(text.encode("utf-8"))
    while len(tokens) >= 2:
        pairs, _ = find_maxpair(tokens)
        smallest_pair = min(pairs, key=lambda p: merges.get(p, float("inf")))
        if smallest_pair not in merges:
            break
        tokens = merge(tokens, smallest_pair, merges[smallest_pair])
    return tokens

s = "Bu bir deneme cümlesidir."
print(encode(s))
print(decode(encode(s)) == s) 

[66, 117, 32, 98, 267, 32, 100, 101, 110, 101, 109, 260, 99, 258, 109, 108, 101, 115, 105, 100, 267, 46]
True
